In [61]:
import sys
sys.path.insert(1, '../../code/')
import os
import pickle
import torch
import numpy as np
from tqdm import tqdm
from datetime import datetime
import tractogramReader as tr
import time
from matplotlib import pyplot as plt
from scipy import stats
import seaborn as sns
import random
import nibabel as nib
from model import ConvAE_256res
from utils import *
from clustering import Kmeans

In [62]:
device = set_device()
RANDOM_SEED = 42
res = 256
data_shape = [3, res]
start = time.time()
LS = torch.load('HCP_Embeddings/HCP_38sub_N256_L1_64ld_latentSpace.pt', map_location=torch.device('cpu'))
LS = torch.reshape(LS,(LS.shape[0], LS.shape[1]))

GPU is enabled. 



In [ ]:
LS_keys = list(np.arange(LS.shape[0]))
LS_dict = {LS_keys[i]: LS[i] for i in tqdm(range(len(LS_keys)))}

In [ ]:
LS_dict_update = LS_dict
clusters = {}
label = 0
while tqdm(LS_dict_update):
    # get the existing keys of the updated list
    keys = list(LS_dict_update.keys())
    # obtain the streamlines from the originl list using the keys and get the distances of the first to the others
    diff = LS[keys[1:],:]-LS[keys[0],:]
    dist = torch.norm(diff, dim=1)
    # keep the keys associated with the 
    #dist_dict = {key: dist[i] for i, key in enumerate(keys[1:]) if dist[i]<500}
    mask = dist < 500

    # Create the filtered keys using NumPy indexing
    filtered_keys = np.array(keys[1:])[mask]

    # Create the `dist_dict` using a dictionary comprehension
    dist_dict = {key: dist[i] for i, key in enumerate(filtered_keys)}

    # Make sure to add the key 0 to the `dist_dict` as well
    dist_dict[keys[0]] = 0

    #dist_dict[keys[0]] = 0

    #keys_to_remove = [key for key, value in dist_dict.items() if value < 500]
    LS_dict_update = {key: value for key, value in LS_dict_update.items() if key not in dist_dict.keys()}
    clusters[label] = dist_dict.keys()
    label += 1


In [60]:
len(clusters)

496